# Composable Conditional Risk in Caller-Owned Models

This walkthrough demonstrates the Feature 016 composition boundary using reviewed, immutable OEL fixtures. It does not fit an SPN, compile a new circuit, regenerate evidence, enumerate decisions, or optimize a benchmark. Instead, it reloads the exact current-SPFlow artifact and max/PWL conditional pairs already validated by automated tests.

The event is violation $Y=1$. For retained decisions $X=x$, OEL exposes two compiled roots:

$$z_{\mathrm{num}}(x) \approx \log p_{X,Y}(x,1),$$

$$z_{\mathrm{den}}(x) \approx \log p_X(x).$$

For continuous $x$, these are **continuous densities, not point probabilities**. Their difference is the authoritative conditional log-risk,

$$z_{\mathrm{num}}(x)-z_{\mathrm{den}}(x)\approx \log P(Y=1\mid X=x),$$

The raw `$z_{\mathrm{num}}(x)-z_{\mathrm{den}}(x)$` is retained without clamping, and exponentiating it gives a diagnostic conditional probability. The denominator is an **explicit learned denominator** from the same fitted SPN, even when the intended decision-sampling design was uniform.

## Ownership boundary

The **caller owns** the primary decision variables, objective, deterministic constraints, $\alpha$, chance constraint, and decision to invoke a solver. **OEL owns** only the namespaced attachment block: the numerator and denominator circuits, auxiliary variables, decision-coordinate links, validity constraints, and categorical exactly-one constraint.

The fixed-assignment examples below are objective-free diagnostic solves. The final policy example only composes a caller objective and constraints around the returned expression; it is intentionally left unsolved.

In [ ]:
import json
import math
from pathlib import Path

import highspy
import pyomo.environ as pyo

from open_energy_lab.tpm import (
    TPMBinaryDecisionMapping,
    TPMCategoricalDecisionMapping,
    TPMCompilerFidelityEvidence,
    TPMContinuousDecisionMapping,
    TPMEmbeddingParityEvidence,
    attach_conditional_mio_pair,
    build_embedding_parity_evidence,
    evaluate_conditional_mio_pair,
    inspect_conditional_mio_attachment,
    load_conditional_pair,
    load_tpm_artifact,
)

FIXTURE_ROOT = Path("tests/fixtures/tpm")
SOURCE_PATH = FIXTURE_ROOT / "conditional_mio_embedding_v1.json"
ARTIFACT_PATH = FIXTURE_ROOT / "tpm_artifact_conditional_mio_v3.json"
MAX_PAIR_PATH = FIXTURE_ROOT / "conditional_pair_max_v1.json"
PWL_PAIR_PATH = FIXTURE_ROOT / "conditional_pair_pwl_v1.json"
EVIDENCE_PATH = FIXTURE_ROOT / "conditional_mio_evidence_v1.json"

## 1. Reload reviewed evidence

The source rows deliberately contain nonuniform evidence for the continuous decision. The persisted schema-v3 artifact retains the declared domains, while both conditional pairs share its identity and differ only in their recorded sum encoding: `max` or `piecewise`. The evidence bundle already contains two embedding-parity records and two native-fidelity records.

In [ ]:
source_fixture = json.loads(SOURCE_PATH.read_text(encoding="utf-8"))
artifact = load_tpm_artifact(ARTIFACT_PATH)
max_pair = load_conditional_pair(MAX_PAIR_PATH)
pwl_pair = load_conditional_pair(PWL_PAIR_PATH)
evidence_bundle = json.loads(EVIDENCE_PATH.read_text(encoding="utf-8"))

loaded_parity_records = tuple(
    TPMEmbeddingParityEvidence.from_dict(record)
    for record in evidence_bundle["embedding_parity_records"]
)
loaded_fidelity_records = tuple(
    TPMCompilerFidelityEvidence.from_dict(record)
    for record in evidence_bundle["compiler_fidelity_records"]
)
assignments_by_case = {
    case["case_name"]: dict(case["values"])
    for case in source_fixture["representative_assignments"]
}

reviewed_inputs = {
    "artifact_schema": artifact.schema_version,
    "artifact_id": artifact.artifact_id,
    "pair_modes": (max_pair.encoding.mode, pwl_pair.encoding.mode),
    "parity_record_count": len(loaded_parity_records),
    "fidelity_record_count": len(loaded_fidelity_records),
}
reviewed_inputs

## 2. Attach all supported mapping forms

Each diagnostic model belongs to the caller and contains: a continuous physical decision `x`, a binary `activation`, and three caller-owned binary indicators ordered as `economy`, `standard`, and `priority`. OEL references those variables rather than replacing them. It links physical $x\in[40,160]$ to the persisted model coordinate, interprets the binary states through their declared ordering, and adds exactly one constraint for the categorical indicators.

We fix just one representative assignment for each encoding. This is a solver-parity diagnostic, not a Cartesian query table.

In [ ]:
CATEGORIES = ("economy", "standard", "priority")


def fixed_assignment_run(pair, case_name, namespace):
    assignments = assignments_by_case[case_name]
    model = pyo.ConcreteModel()
    model.x = pyo.Var(bounds=(40.0, 160.0))
    model.activation = pyo.Var(domain=pyo.Binary)
    model.service_tier = pyo.Var(CATEGORIES, domain=pyo.Binary)

    attachment = attach_conditional_mio_pair(
        pair,
        model,
        mappings=(
            TPMContinuousDecisionMapping("x", model.x),
            TPMBinaryDecisionMapping("activation", model.activation),
            TPMCategoricalDecisionMapping(
                "service_tier",
                tuple((label, model.service_tier[label]) for label in CATEGORIES),
            ),
        ),
        namespace=namespace,
    )

    model.x.fix(float(assignments["x"]))
    model.activation.fix(int(assignments["activation"] == "on"))
    for label in CATEGORIES:
        model.service_tier[label].fix(int(label == assignments["service_tier"]))

    direct = evaluate_conditional_mio_pair(
        pair, declared_domain_assignments=assignments
    )
    solver = pyo.SolverFactory("highs")
    solver.options["threads"] = 1
    result = solver.solve(model)
    attached = inspect_conditional_mio_attachment(
        attachment,
        solver_metadata={
            "solver_name": "highs",
            "solver_version": highspy.Highs().version(),
            "status": str(result.solver.status),
            "termination_condition": str(result.solver.termination_condition),
        },
    )
    parity = build_embedding_parity_evidence(
        pair,
        case_results={case_name: (direct, attached)},
        provenance={"consumer": "conditional-mio-walkthrough"},
    )
    return {
        "model": model,
        "attachment": attachment,
        "direct": direct,
        "attached": attached,
        "parity": parity,
    }

In [ ]:
max_run = fixed_assignment_run(max_pair, "lower-boundary", "max_risk")
pwl_run = fixed_assignment_run(pwl_pair, "interior", "pwl_risk")
max_parity = max_run["parity"]
pwl_parity = pwl_run["parity"]

fixed_assignment_summary = {
    "max": {
        "case": "lower-boundary",
        "accepted": max_parity.accepted,
        "direct_log_risk": max_run["direct"].conditional_log_risk,
        "attached_log_risk": max_run["attached"].conditional_log_risk,
    },
    "piecewise": {
        "case": "interior",
        "accepted": pwl_parity.accepted,
        "direct_log_risk": pwl_run["direct"].conditional_log_risk,
        "attached_log_risk": pwl_run["attached"].conditional_log_risk,
    },
}
fixed_assignment_summary

## 3. Keep two comparisons separate

**Embedding/solver parity** compares a Pyomo/HiGHS attachment with direct evaluation of the *identical compiled max or PWL semantics*. Root and log-risk errors plus attachment-local residuals are acceptance-gated at the persisted tolerance. A failure here means the symbolic embedding or solve did not reproduce its own circuit.

**Native/compiler fidelity** compares direct compiled values with native current-SPFlow numerator and denominator queries. Those signed differences measure compiler approximation, so they are retained as scientific evidence with no adequacy gate. Max and PWL may disagree with native SPFlow while still passing embedding parity.

In [ ]:
embedding_solver_summary = tuple(
    {
        "mode": record.encoding["mode"],
        "accepted": record.accepted,
        "maximum_log_risk_error": max(
            case["absolute_errors"]["conditional_log_risk"]
            for case in record.cases
        ),
    }
    for record in loaded_parity_records
)
native_compiler_summary = tuple(
    {
        "mode": record.encoding["mode"],
        "numerical_gate": record.metric_definitions["numerical_gate"],
        "conditional_log_risk_differences": tuple(
            case["differences"]["conditional_log_risk"]
            for case in record.cases
        ),
    }
    for record in loaded_fidelity_records
)
{
    "embedding_solver_parity": embedding_solver_summary,
    "native_compiler_fidelity": native_compiler_summary,
}

## 4. Compose an unsolved caller policy

The final model demonstrates the intended reuse boundary. The physical decision is the affine expression `base_order + 2 * reserve`; the remaining mappings use a caller binary and ordered one-hot indicators. After attachment, the caller adds an unrelated deterministic constraint, its own $\alpha$, the chance constraint

$$z_{\mathrm{num}}(x)-z_{\mathrm{den}}(x) \leq \log \alpha,$$

and its own objective. OEL does not choose the threshold, add that constraint, or invoke the solver. This notebook likewise leaves the policy unsolved: it demonstrates composition, not a benchmark recommendation.

In [ ]:
policy_model = pyo.ConcreteModel()
policy_model.base_order = pyo.Var(bounds=(40.0, 150.0))
policy_model.reserve = pyo.Var(bounds=(0.0, 5.0))
policy_model.activation = pyo.Var(domain=pyo.Binary)
policy_model.service_tier = pyo.Var(CATEGORIES, domain=pyo.Binary)

policy_attachment = attach_conditional_mio_pair(
    max_pair,
    policy_model,
    mappings=(
        TPMContinuousDecisionMapping(
            "x", policy_model.base_order + 2.0 * policy_model.reserve
        ),
        TPMBinaryDecisionMapping("activation", policy_model.activation),
        TPMCategoricalDecisionMapping(
            "service_tier",
            tuple(
                (label, policy_model.service_tier[label])
                for label in CATEGORIES
            ),
        ),
    ),
    namespace="policy_risk",
)

policy_model.caller_deterministic_constraint = pyo.Constraint(
    expr=policy_model.base_order + policy_model.reserve <= 150.0
)
policy_model.caller_alpha = pyo.Param(initialize=0.6, mutable=True)
policy_model.caller_chance_constraint = pyo.Constraint(
    expr=policy_attachment.conditional_log_risk
    <= math.log(pyo.value(policy_model.caller_alpha))
)
policy_model.caller_objective = pyo.Objective(
    expr=policy_model.base_order
    + 3.0 * policy_model.reserve
    + 2.0 * policy_model.activation
)

policy_unsolved = len(policy_model.solutions) == 0
conditional_demo_complete = (
    max_parity.accepted
    and pwl_parity.accepted
    and all(record.accepted for record in loaded_parity_records)
    and policy_unsolved
)
{
    "attachment_namespace": policy_attachment.namespace,
    "caller_components": (
        "caller_objective",
        "caller_deterministic_constraint",
        "caller_alpha",
        "caller_chance_constraint",
    ),
    "policy_unsolved": policy_unsolved,
    "conditional_demo_complete": conditional_demo_complete,
}